# Avance 2 – Clasificador de documentos (visión por computadora)
**Proyecto:** Asistente de reclutamiento del restaurante El Sauce
**Alumna:** Juana Paola Puente Huerta

Este cuaderno entrena una red neuronal convolucional (CNN) con **transferencia de aprendizaje (EfficientNetB0)**
para reconocer 4 clases de imágenes que un candidato puede enviar al chatbot:

| Clase | Descripción |
|---|---|
| `ine` | Credencial para votar (frente o reverso) |
| `licencia` | Licencia de conducir |
| `comprobante` | Recibo de luz (CFE) o de agua |
| `otro` | Cualquier imagen que no sea uno de los documentos anteriores |

> **Antes de ejecutar:** ve a *Entorno de ejecución → Cambiar tipo de entorno de ejecución* y elige **GPU (T4)**.
> El entrenamiento tarda unos minutos con GPU.

## 1. Instalación y configuración

In [ ]:
!pip install -q pillow-heif pymupdf

import os, json, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
import fitz  # PyMuPDF, para convertir PDF a imagen
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

register_heif_opener()   # permite abrir fotos de iPhone (.HEIC)

SEMILLA = 42             # para que los resultados se puedan repetir
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUTA_DATOS  = '/content/drive/MyDrive/ChatbotRH/documentos'
RUTA_MODELO = '/content/drive/MyDrive/ChatbotRH/modelo'
os.makedirs(RUTA_MODELO, exist_ok=True)

CLASES = ['comprobante', 'ine', 'licencia', 'otro']
TAM = 224   # tamaño de entrada de EfficientNetB0

## 2. Carga y preparación de las imágenes
Cada archivo se convierte a una imagen RGB de 224×224 píxeles:
- Las fotos se giran según su orientación original (EXIF), porque el celular a veces las guarda de lado.
- Los PDF se convierten a imagen usando su primera página.
- La imagen se ajusta al cuadro de 224×224 **sin deformarla**, rellenando los bordes.

In [ ]:
EXT_IMAGEN = ('.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp', '.bmp')

def cargar_imagen(ruta, tam=TAM):
    """Abre una foto o un PDF y la regresa como arreglo RGB de tam x tam."""
    if ruta.lower().endswith('.pdf'):
        pagina = fitz.open(ruta)[0]
        pix = pagina.get_pixmap(dpi=100)
        img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
    else:
        img = Image.open(ruta)
        img = ImageOps.exif_transpose(img)   # corrige fotos giradas
        img = img.convert('RGB')
    img = ImageOps.pad(img, (tam, tam), color=(0, 0, 0))   # sin deformar
    return np.array(img, dtype=np.float32)

X, y, rutas = [], [], []
for idx, clase in enumerate(CLASES):
    carpeta = os.path.join(RUTA_DATOS, clase)
    for archivo in sorted(os.listdir(carpeta)):
        if not archivo.lower().endswith(EXT_IMAGEN + ('.pdf',)):
            continue
        ruta = os.path.join(carpeta, archivo)
        try:
            X.append(cargar_imagen(ruta)); y.append(idx); rutas.append(ruta)
        except Exception as e:
            print("No se pudo leer:", archivo, "->", e)

X = np.stack(X); y = np.array(y)
print("Total de imágenes:", len(X), "| forma:", X.shape)
for i, c in enumerate(CLASES):
    print(f"  {c:<12} {np.sum(y == i)}")

### Ejemplos de cada clase

In [ ]:
fig, ejes = plt.subplots(len(CLASES), 5, figsize=(13, 2.8 * len(CLASES)))
for i, clase in enumerate(CLASES):
    indices = np.where(y == i)[0]
    muestra = np.random.choice(indices, min(5, len(indices)), replace=False)
    for j in range(5):
        ax = ejes[i, j]; ax.axis('off')
        if j < len(muestra):
            ax.imshow(X[muestra[j]].astype('uint8'))
        if j == 0:
            ax.set_title(clase, loc='left', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. División en entrenamiento, validación y prueba
- **Entrenamiento (70 %)**: imágenes con las que aprende el modelo.
- **Validación (15 %)**: para vigilar el aprendizaje y detenerlo a tiempo.
- **Prueba (15 %)**: imágenes que el modelo **nunca vio**; con ellas se calculan las métricas finales.

La división es **estratificada**: cada parte conserva la misma proporción de clases.

In [ ]:
X_temp, X_test, y_temp, y_test, r_temp, r_test = train_test_split(
    X, y, rutas, test_size=0.15, stratify=y, random_state=SEMILLA)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=SEMILLA)   # 0.1765 * 0.85 ≈ 0.15

for nombre, conj in [('Entrenamiento', y_train), ('Validación', y_val), ('Prueba', y_test)]:
    conteo = {CLASES[i]: int(np.sum(conj == i)) for i in range(len(CLASES))}
    print(f"{nombre:<14} {len(conj):>3}  {conteo}")

# Pesos por clase: compensan que algunas clases tengan menos imágenes
pesos = compute_class_weight('balanced', classes=np.arange(len(CLASES)), y=y_train)
PESOS_CLASE = dict(enumerate(pesos))
print("\nPesos por clase:", {CLASES[k]: round(v, 2) for k, v in PESOS_CLASE.items()})

## 4. Arquitectura del modelo
1. **Aumento de datos**: en cada época, las imágenes de entrenamiento se giran, acercan, mueven y cambian de brillo y contraste al azar.
   Así el modelo ve variaciones nuevas y no memoriza las fotos. *No se usa volteo horizontal, porque el texto quedaría al revés.*
2. **EfficientNetB0 preentrenada con ImageNet** (sin su capa final): extrae las características visuales (bordes, colores, logos, formas).
3. **Capa de clasificación nueva**: promedio global, *dropout* y una capa densa con 4 salidas (softmax).

In [ ]:
aumento = keras.Sequential([
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name='aumento_de_datos')

base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(TAM, TAM, 3))
base.trainable = False   # fase 1: la base queda congelada

entrada = keras.Input(shape=(TAM, TAM, 3))
x = aumento(entrada)
x = base(x, training=False)          # EfficientNet ya incluye su propio escalado de píxeles
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
salida = layers.Dense(len(CLASES), activation='softmax')(x)
modelo = keras.Model(entrada, salida, name='clasificador_documentos')

modelo.summary(show_trainable=True)

## 5. Entrenamiento
**Fase 1 – Extracción de características:** solo se entrena la capa nueva; EfficientNet se mantiene congelada.
**Fase 2 – Ajuste fino (*fine-tuning*):** se descongelan las últimas capas de EfficientNet y se entrenan con una tasa de aprendizaje muy baja,
para adaptarlas a nuestros documentos sin perder lo que ya sabían.

In [ ]:
parar = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

modelo.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])

hist1 = modelo.fit(X_train, y_train, validation_data=(X_val, y_val),
                   epochs=25, batch_size=16, class_weight=PESOS_CLASE, callbacks=[parar])

In [ ]:
# Fase 2: ajuste fino de las últimas 30 capas (las BatchNormalization se dejan congeladas)
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False
for capa in base.layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False

modelo.compile(optimizer=keras.optimizers.Adam(1e-5),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])

hist2 = modelo.fit(X_train, y_train, validation_data=(X_val, y_val),
                   epochs=15, batch_size=16, class_weight=PESOS_CLASE, callbacks=[parar])

### Curvas de aprendizaje

In [ ]:
def unir(h1, h2, clave):
    return h1.history[clave] + h2.history[clave]

ep_fase1 = len(hist1.history['loss'])
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metrica, titulo in [(ejes[0], 'accuracy', 'Exactitud (accuracy)'), (ejes[1], 'loss', 'Pérdida (loss)')]:
    ax.plot(unir(hist1, hist2, metrica), label='Entrenamiento')
    ax.plot(unir(hist1, hist2, 'val_' + metrica), label='Validación')
    ax.axvline(ep_fase1 - 0.5, color='gray', linestyle='--', label='Inicio ajuste fino')
    ax.set_title(titulo); ax.set_xlabel('Época'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Evaluación con el conjunto de prueba
Se calculan las métricas que pide el proyecto:
- **Accuracy**: porcentaje de imágenes clasificadas correctamente.
- **Precision**: de las imágenes que el modelo dijo que eran de una clase, cuántas sí lo eran.
- **Recall**: de las imágenes que sí eran de una clase, cuántas encontró el modelo.
- **F1-score**: promedio armónico entre precision y recall.
- **Matriz de confusión**: muestra en qué clases se equivoca y con cuáles las confunde.

In [ ]:
probs = modelo.predict(X_test, verbose=0)
y_pred = probs.argmax(axis=1)

print(f"Accuracy en prueba: {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred, target_names=CLASES, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=range(len(CLASES)))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title('Matriz de confusión (conjunto de prueba)')
ax.set_xlabel('Predicción'); ax.set_ylabel('Clase real')
plt.tight_layout(); plt.show()

### Predicciones sobre las imágenes de prueba

In [ ]:
n = len(X_test)
cols = 5; filas = int(np.ceil(n / cols))
fig, ejes = plt.subplots(filas, cols, figsize=(14, 3 * filas))
for i, ax in enumerate(np.array(ejes).ravel()):
    ax.axis('off')
    if i >= n: continue
    ax.imshow(X_test[i].astype('uint8'))
    ok = y_pred[i] == y_test[i]
    ax.set_title(f"Real: {CLASES[y_test[i]]}\nPred: {CLASES[y_pred[i]]} ({probs[i].max():.0%})",
                 fontsize=9, color='green' if ok else 'red')
plt.tight_layout(); plt.show()

errores = [os.path.basename(r_test[i]) for i in range(n) if y_pred[i] != y_test[i]]
print("Imágenes mal clasificadas:", errores if errores else "ninguna")

## 7. Guardar el modelo
El modelo se guarda en Drive para que el chatbot lo use después sin volver a entrenar.

In [ ]:
modelo.save(os.path.join(RUTA_MODELO, 'modelo_documentos.keras'))
with open(os.path.join(RUTA_MODELO, 'clases.json'), 'w') as f:
    json.dump(CLASES, f)
print("Modelo guardado en", RUTA_MODELO)

## 8. Función para clasificar un documento nuevo
Esta es la función que usará el chatbot. Si la confianza del modelo es menor al umbral,
el resultado se marca como **dudoso** para pedirle al candidato una foto más clara.

In [ ]:
UMBRAL = 0.70

def clasificar_documento(ruta, umbral=UMBRAL):
    """Regresa (clase, confianza). Si la confianza es baja, la clase es 'dudoso'."""
    img = cargar_imagen(ruta)[None, ...]
    p = modelo.predict(img, verbose=0)[0]
    clase, confianza = CLASES[int(p.argmax())], float(p.max())
    if confianza < umbral:
        return 'dudoso', confianza
    return clase, confianza

# Prueba rápida con una imagen de prueba
ejemplo = r_test[0]
print(os.path.basename(ejemplo), '->', clasificar_documento(ejemplo))